# Revisión capa GOLD (marts Power BI + feature stores ML)

Los 6 marts para dashboards usan **grano agregado** (resúmenes por
fecha/hora/zona/servicio); el detalle viaje a viaje vive en silver/star/facts.
**Prueba de no-pérdida**: `SUM(viajes)` de cada mart == filas de los facts, exacto.

In [1]:
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq

DATA = Path("../data") if Path("../data").exists() else Path("data")
CATS = ["green", "yellow", "fhv", "fhvhv"]
YEARS = [2023, 2024, 2025]
MESES = [(y, m) for y in YEARS for m in range(1, 13)]
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(25)


def parquet_rows(path: Path) -> int | None:
    """Filas totales de un archivo/directorio parquet leyendo solo metadatos.

    Ojo: Spark escribe "archivos" mensuales como DIRECTORIOS llamados *.parquet
    con part-files dentro — por eso se filtra con is_file(). Se ignora la
    basura _temporary y los archivos de 0 bytes que deja un job interrumpido.
    """
    if not path.exists():
        return None
    files = (
        [path]
        if path.is_file()
        else [
            f for f in path.rglob("*.parquet")
            if f.is_file() and f.stat().st_size > 0 and "_temporary" not in str(f)
        ]
    )
    if not files:
        return None
    return sum(pq.ParquetFile(f).metadata.num_rows for f in files)


def dir_mb(path: Path) -> float:
    """Peso en MB de un archivo o directorio."""
    if not path.exists():
        return 0.0
    files = [path] if path.is_file() else [f for f in path.rglob("*") if f.is_file()]
    return round(sum(f.stat().st_size for f in files) / 1024**2, 2)

## Inventario de salidas gold

In [2]:
inventario = []
for sub in ("marts", "ml"):
    base = DATA / "gold" / sub
    for d in sorted(p for p in base.iterdir() if p.is_dir()):
        inventario.append({"tipo": sub, "tabla": d.name, "filas": parquet_rows(d), "peso_mb": dir_mb(d)})
inv = pl.DataFrame(inventario)
print(f"gold total: {inv['peso_mb'].sum():,.0f} MB")
inv

gold total: 5,508 MB


tipo,tabla,filas,peso_mb
str,str,i64,f64
"""marts""","""mart_abc_xyz_zones""",2335,0.11
"""marts""","""mart_demand_volume""",20494085,154.59
"""marts""","""mart_financial_performance""",2704556,91.95
"""marts""","""mart_operational_profile""",2704556,54.07
"""marts""","""mart_supply_demand_balance""",26550652,174.36
"""marts""","""mart_tipping_behavior""",501341,7.04
"""ml""","""kmodes_model""",293063,3.08
"""ml""","""ml_feat_arima_trips""",607044,3.62
"""ml""","""ml_feat_isolation_fraud""",128416505,3270.57


## Cuadre de conteos: cada viaje contado exactamente una vez

financial/operational/tipping excluyen `fhv` **por diseño** (esa categoría
no publica tarifas ni distancias), por eso suman 292M y no 317M.

In [3]:
MART_CATS = {
    "mart_demand_volume": CATS,
    "mart_financial_performance": ["green", "yellow", "fhvhv"],
    "mart_operational_profile": ["green", "yellow", "fhvhv"],
    "mart_tipping_behavior": ["green", "yellow", "fhvhv"],
}
cuadre = []
for mart, cats in MART_CATS.items():
    suma = (pl.scan_parquet(str(DATA / "gold" / "marts" / mart / "**" / "*.parquet"), hive_partitioning=True)
              .select(pl.col("viajes").sum()).collect().item())
    facts = sum(parquet_rows(DATA / "silver" / "star" / "facts" / f"fact_{c}_trip") or 0 for c in cats)
    cuadre.append({"mart": mart, "sum_viajes": suma, "filas_facts": facts, "cuadra": suma == facts})
pl.DataFrame(cuadre)

mart,sum_viajes,filas_facts,cuadra
str,i64,i64,bool
"""mart_demand_volume""",902314793,902314793,true
"""mart_financial_performance""",843928141,843928141,true
"""mart_operational_profile""",843928141,843928141,true
"""mart_tipping_behavior""",843928141,843928141,true


## Vista previa de cada mart

In [4]:
for mart in sorted((DATA / "gold" / "marts").iterdir()):
    if not mart.is_dir():
        continue
    df = pl.read_parquet(str(mart / "**" / "*.parquet"), hive_partitioning=True, n_rows=3)
    print(f"\n=== {mart.name} ({len(df.columns)} columnas) ===")
    print(df.head(3))


=== mart_abc_xyz_zones (12 columnas) ===
shape: (3, 12)
┌────────┬────────┬────────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬──────┐
│ pu_loc ┆ boroug ┆ zone   ┆ ingres ┆ viaje ┆ viaje ┆ coefi ┆ clase ┆ porce ┆ clase ┆ servi ┆ year │
│ ation_ ┆ h      ┆ ---    ┆ os_tot ┆ s_dia ┆ s_dia ┆ cient ┆ _xyz  ┆ ntaje ┆ _abc  ┆ ce_id ┆ ---  │
│ id     ┆ ---    ┆ str    ┆ ales_z ┆ rios_ ┆ rios_ ┆ e_var ┆ ---   ┆ _acum ┆ ---   ┆ ---   ┆ i64  │
│ ---    ┆ str    ┆        ┆ ona    ┆ prome ┆ std   ┆ iacio ┆ str   ┆ ulado ┆ str   ┆ str   ┆      │
│ f64    ┆        ┆        ┆ ---    ┆ dio   ┆ ---   ┆ n_xyz ┆       ┆ _ingr ┆       ┆       ┆      │
│        ┆        ┆        ┆ f64    ┆ ---   ┆ f64   ┆ ---   ┆       ┆ esos  ┆       ┆       ┆      │
│        ┆        ┆        ┆        ┆ f64   ┆       ┆ f64   ┆       ┆ ---   ┆       ┆       ┆      │
│        ┆        ┆        ┆        ┆       ┆       ┆       ┆       ┆ f64   ┆       ┆       ┆      │
╞════════╪════════╪════════╪══════

## Análisis de ejemplo 1: demanda mensual por servicio (millones de viajes)

In [5]:
demanda = (pl.scan_parquet(str(DATA / "gold" / "marts" / "mart_demand_volume" / "**" / "*.parquet"),
                           hive_partitioning=True)
    .group_by("service_id", "month").agg((pl.col("viajes").sum() / 1e6).round(2).alias("millones"))
    .collect()
    .pivot(values="millones", index="month", on="service_id")
    .sort("month"))
demanda

month,fhvhv,yellow,fhv,green
i64,f64,f64,f64,f64
1,58.55,9.39,4.29,0.17
2,56.66,9.38,3.85,0.16
3,62.23,10.99,4.97,0.18
4,58.63,10.63,4.38,0.17
5,61.64,11.67,4.94,0.18
6,59.36,11.02,4.83,0.17
7,57.97,9.73,4.93,0.16
8,56.72,9.22,5.17,0.16
9,58.5,10.56,5.15,0.17


## Análisis de ejemplo 2: top 10 zonas por demanda anual

In [6]:
(pl.scan_parquet(str(DATA / "gold" / "marts" / "mart_demand_volume" / "**" / "*.parquet"),
                 hive_partitioning=True)
   .group_by("pu_zone", "pu_borough").agg(pl.col("viajes").sum().alias("viajes"))
   .sort("viajes", descending=True).head(10).collect())

pu_zone,pu_borough,viajes
str,str,i64
null,null,46834393
"""JFK Airport""","""Queens""",18718495
"""LaGuardia Airport""","""Queens""",17996123
"""Midtown Center""","""Manhattan""",14306684
"""Times Sq/Theatre District""","""Manhattan""",12948529
"""East Village""","""Manhattan""",12487216
"""Upper East Side South""","""Manhattan""",11668398
"""East Chelsea""","""Manhattan""",11318432
"""Union Sq""","""Manhattan""",11043692


## Feature stores ML

- `ml_feat_isolation_fraud`: yellow+green **completos** (el modelo puntúa fraude viaje a viaje).
- `ml_feat_kmodes_trips`: yellow+green completos + **muestra 5% de fhvhv** (el modelo K-Modes entrena con máx. 100k filas por servicio; la muestra es 120× eso).
- `ml_feat_arima_trips`: serie agregada zona×hora para pronóstico.

In [7]:
# Preview solo de los 3 FEATURE STORES (ml_feat_*). Los directorios de
# salidas de modelos (--gold-ml: kmodes_model, ml_isolation_fraud_scores,
# ml_sarimax_trips_forecast) usan otros esquemas de particion y se listan aparte.
for store in sorted((DATA / "gold" / "ml").glob("ml_feat_*")):
    if not store.is_dir():
        continue
    df = pl.read_parquet(str(store / "**" / "*.parquet"), hive_partitioning=True, n_rows=3)
    print(f"\n=== {store.name}: {parquet_rows(store):,} filas | {dir_mb(store):,.0f} MB ===")
    print(df.head(3))

print("\n--- salidas de modelos entrenados (--gold-ml) ---")
for d in sorted((DATA / "gold" / "ml").iterdir()):
    if d.is_dir() and not d.name.startswith("ml_feat_"):
        n = parquet_rows(d)
        print(f"{d.name}: {n:,} filas | {dir_mb(d):,.1f} MB" if n else f"{d.name}: {dir_mb(d):,.1f} MB")


=== ml_feat_arima_trips: 607,044 filas | 4 MB ===
shape: (3, 10)
┌───────────┬───────────┬──────────┬──────────┬─────┬──────────┬──────────┬─────────┬──────┬───────┐
│ service_i ┆ pickup_ho ┆ trip_cou ┆ hour_of_ ┆ dow ┆ is_weeke ┆ is_holid ┆ borough ┆ year ┆ month │
│ d         ┆ ur        ┆ nt       ┆ day      ┆ --- ┆ nd       ┆ ay       ┆ ---     ┆ ---  ┆ ---   │
│ ---       ┆ ---       ┆ ---      ┆ ---      ┆ i32 ┆ ---      ┆ ---      ┆ str     ┆ i64  ┆ i64   │
│ str       ┆ datetime[ ┆ i64      ┆ i32      ┆     ┆ bool     ┆ bool     ┆         ┆      ┆       │
│           ┆ ns]       ┆          ┆          ┆     ┆          ┆          ┆         ┆      ┆       │
╞═══════════╪═══════════╪══════════╪══════════╪═════╪══════════╪══════════╪═════════╪══════╪═══════╡
│ green     ┆ 2023-01-1 ┆ 2        ┆ 15       ┆ 2   ┆ false    ┆ false    ┆ Bronx   ┆ 2023 ┆ 1     │
│           ┆ 7         ┆          ┆          ┆     ┆          ┆          ┆         ┆      ┆       │
│           ┆ 20:00:00  ┆